# 2. Data Collection with Web Scraping
**Goal:** scrape the Wikipedia page *List of Falcon 9 and Falcon Heavy launches* (frozen revision) into a dataframe.

Flow: request page -> parse with BeautifulSoup -> find launch tables -> loop over rows and extract cells -> dataframe -> CSV.

In [ ]:
import requests, re, pandas as pd
from bs4 import BeautifulSoup

In [ ]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
response = requests.get(static_url, headers={'User-Agent': 'Mozilla/5.0'})
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title.string)

## Find the launch tables

In [ ]:
tables = soup.find_all('table', 'wikitable plainrowheaders collapsible')
print(len(tables), 'launch tables found')
# the column names live in the header row of the 3rd table
header = [th.get_text(' ', strip=True) for th in tables[2].find_all('th')]
print(header[:15])

## Extract the launch records row by row

In [ ]:
def clean(cell):
    return re.sub(r'\[.*?\]', '', cell.get_text(' ', strip=True)).strip()

records = []
for table in tables:
    for row in table.find_all('tr'):
        cells = row.find_all(['th', 'td'])
        # a launch row starts with a numeric flight number in its first cell
        if len(cells) < 8 or not clean(cells[0]).isdigit():
            continue
        text = [clean(c) for c in cells]
        records.append({
            'Flight No.': text[0],
            'Date and time': text[1],
            'Version, Booster': text[2],
            'Launch site': text[3],
            'Payload': text[4],
            'Payload mass': text[5],
            'Orbit': text[6],
            'Customer': text[7],
            'Launch outcome': text[8] if len(text) > 8 else '',
            'Booster landing': text[9] if len(text) > 9 else '',
        })
launch_dict = pd.DataFrame(records)
print(launch_dict.shape)
launch_dict.head()

In [ ]:
launch_dict.to_csv('spacex_web_scraped.csv', index=False)
launch_dict['Booster landing'].value_counts()

> If the number of cells per row differs on your page version, check `print(text)` for one row and adjust the column positions above.